# GPT5-mini text generation

### Import libraries

In [1]:
import numpy as np
import pandas as pd

import os
import json

from google.colab import drive, userdata
from openai import OpenAI

### Set up Google Drive mounting and define input/output paths:

In [2]:
# Mount the google drive folder
drive.mount('/content/drive', force_remount = True) # force reconnection

# Define directory, input, and output paths
project_directory = "/content/drive/MyDrive/Colab Notebooks/DS266/final_project/266_group_project"
input_file = os.path.join(project_directory, "all.jsonl")
output_file = os.path.join(project_directory, "gpt5mini data/gpt5mini_original.jsonl")

# Ensure the output directory exists
if not os.path.exists(project_directory):
  os.makedirs(project_directory)

# Get access to the seceret api key
os.environ["OPENAI_API_KEY"] = userdata.get('HC3_GPT5mini')
client = OpenAI()

Mounted at /content/drive


### Read the input file:

In [3]:
# JSONL - each line is a json object
df = pd.read_json(input_file, lines=True)
print('Sucessfully read the HC3 raw file!')

print(f'The raw file contains {len(df)} records/rows.')
print(f'Column names: {list(df.columns)}')

print('\nThe first 5 rows:')
display(df.head())

Sucessfully read the HC3 raw file!
The raw file contains 24322 records/rows.
Column names: ['question', 'human_answers', 'chatgpt_answers', 'index', 'source']

The first 5 rows:


,question,human_answers,chatgpt_answers,index,source
0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,NaN,reddit_eli5
1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,NaN,reddit_eli5
2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,NaN,reddit_eli5
3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,NaN,reddit_eli5
4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,NaN,reddit_eli5


### Print the sample question and GPT-3.5 answer:

In [4]:
print('Sample question:')
display(df['question'][0])
print('\nSample GPT-3.5 answer:')
display(df['chatgpt_answers'][0])

Sample question:


'Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is every book I hear about a " NY Times # 1 Best Seller " ? Should n\'t there only be one " # 1 " best seller ? Please explain like I\'m five.'


Sample GPT-3.5 answer:


['There are many different best seller lists that are published by various organizations, and the New York Times is just one of them. The New York Times best seller list is a weekly list that ranks the best-selling books in the United States based on sales data from a number of different retailers. The list is published in the New York Times newspaper and is widely considered to be one of the most influential best seller lists in the book industry. \nIt\'s important to note that the New York Times best seller list is not the only best seller list out there, and there are many other lists that rank the top-selling books in different categories or in different countries. So it\'s possible that a book could be a best seller on one list but not on another. \nAdditionally, the term "best seller" is often used more broadly to refer to any book that is selling well, regardless of whether it is on a specific best seller list or not. So it\'s possible that you may hear about a book being a "bes

### Drop rows that have missing ChatGPT-3.5 answers and duplicate questions

In [5]:
# Drop rows that have missing ChatGPT-3.5 answers
df = df[df['chatgpt_answers'].apply(lambda x: len(x) > 0)].reset_index(drop = True)
print('Number of records/rows after dropping missing GPT-3.5 answers:', len(df))

# Drop the rows that have duplicate questions
df = df.drop_duplicates(subset = ['question']).reset_index(drop = True)
print('Number of records/rows after dropping duplicate questions:', len(df))

print('\nThe first 5 rows:')
display(df.head())

Number of records/rows after dropping missing GPT-3.5 answers: 23867
Number of records/rows after dropping duplicate questions: 23339

The first 5 rows:


,question,human_answers,chatgpt_answers,index,source
0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,NaN,reddit_eli5
1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,NaN,reddit_eli5
2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,NaN,reddit_eli5
3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,NaN,reddit_eli5
4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,NaN,reddit_eli5


### Compute the average lengths of GPT-3.5 answers (used to limit GPT-5mini output lengths)

In [6]:
# GPT-5mini uses a tokenizer that each word is usually equal to ~1.3 tokens
avg_tokens_per_word_gpt5mini = 1.3

avg_lengths_per_answer = df['chatgpt_answers'].apply(lambda ans: len(ans[0].split())).mean()
avg_tokens_per_answer = avg_lengths_per_answer * avg_tokens_per_word_gpt5mini

print(f'Average word count per GPT-3.5 answer: {avg_lengths_per_answer}')
print(f'Average tokens per word for GPT5-mini: {avg_tokens_per_answer}')

Average word count per GPT-3.5 answer: 177.91387805818587
Average tokens per word for GPT5-mini: 231.28804147564165


### Generate GPT5-mini answers in the new column `gpt5mini_answer`:

In [7]:
# Look for where the generation was left off

last_row_id = -1

if os.path.exists(output_file):
  output_df = pd.read_json(output_file, lines = True) # JSONL
  last_row_id = output_df.index[-1]

print('No output file exists.' if last_row_id < 0 else f'Last row generated (row id): {last_row_id}')

Last row generated (row id): 23338


In [8]:
# Start appending to the file from where it's left off

with open(output_file, 'a') as f:

  # Keep track no-answer rows
  no_answer = 0

  # Iterate through each record/row
  for i, row in df.iterrows():

    # Print that the generation is already done if no rows are left to be generated
    if i == len(df) - 1:
      print(f'GPT5-mini {len(df)} answer generations already completed!')

    # Continue if current row is less than or equal to last_row_id
    if i <= last_row_id:
      continue

    # Convert each row to dictionary becasue the output file is going to be json
    row = row.to_dict()

    # Get the question from the dataframe
    question = row['question']

    # Generate response from GPT-5mini
    try:
      gpt5mini_response = client.chat.completions.create(model = 'gpt-5-mini',                                      # We are using GPT-5mini in addition to GPT-3.5
                                                         messages = [{'role': 'user', 'content': question}],
                                                         max_completion_tokens = 1024,
                                                         reasoning_effort = 'low')                                  # Increase efficiency and comparability between models
      gpt5mini_answer = gpt5mini_response.choices[0].message.content

    except Exception as e:
      print(f'Error when processing row {i + 1} (index + 1) question {question}: {e}')
      no_answer += 1
      gpt5mini_answer = None

    # Insert the GPT-5mini answers into the dictionary (each row has a new column in the output file)
    row['gpt5mini_answer'] = gpt5mini_answer

    # Write the row with new column into the output file
    f.write(json.dumps(row) + '\n')   # '\n' so that each row starts with a new line


    # Print out the progress when generating
    if i % 100 == 0:

      # Force writing to the os in case the program crashes
      f.flush()
      os.fsync(f.fileno())

      # Print the progress and saving status
      if i != 0 and (i - last_row_id) > 100:
        print('Finished and saved!')
      print(f'Start generating 100 answers for batch {(i // 100) + 1} ....', end = ' ')

    elif i == len(df) - 1:
      print(f'Finished {len(df)} answer generations!')

GPT5-mini 23339 answer generations already completed!


In [9]:
# Convert JSONL to CSV
out_df = pd.read_json(output_file, lines = True)

output_file_csv = os.path.join(project_directory, 'gpt5mini data/gpt5mini_original.csv')
out_df.to_csv(output_file_csv, index = False)

### Read the output file after answer generatioins:

In [10]:
out_df = pd.read_csv(output_file_csv)
print('Sucessfully read the output file with GPT5-mini answers!')

print(f'The output file contains {len(out_df)} records/rows.')
print(f'New column names: {list(out_df.columns)}')

print('\nThe first 5 rows:')
display(out_df.head())

Sucessfully read the output file with GPT5-mini answers!
The output file contains 23339 records/rows.
New column names: ['question', 'human_answers', 'chatgpt_answers', 'index', 'source', 'gpt5mini_answer']

The first 5 rows:


,question,human_answers,chatgpt_answers,index,source,gpt5mini_answer
0,"Why is every book I hear about a "" NY Times # ...","['Basically there are many categories of "" Bes...",['There are many different best seller lists t...,NaN,reddit_eli5,Great question — it can be confusing! Here’s a...
1,"If salt is so bad for cars , why do we use it ...",['salt is good for not dying in car crashes an...,"[""Salt is used on roads to help melt ice and s...",NaN,reddit_eli5,Think of ice on the road like a big frozen pud...
2,Why do we still have SD TV channels when HD lo...,"[""The way it works is that old TV stations got...","[""There are a few reasons why we still have SD...",NaN,reddit_eli5,Imagine TV channels are like two kinds of pict...
3,Why has nobody assassinated Kim Jong - un He i...,"[""You ca n't just go around assassinating the ...",['It is generally not acceptable or ethical to...,NaN,reddit_eli5,"Okay — I’ll explain it like you asked, simple ..."
4,How was airplane technology able to advance so...,['Wanting to kill the shit out of Germans driv...,['After the Wright Brothers made the first pow...,NaN,reddit_eli5,Okay — imagine a kid who builds a toy boat tha...


### Print the sample question and GPT5-mini answer:

In [11]:
print('Sample question:')
display(out_df['question'][len(out_df) - 2]) # The last answer is NaN, so sample the second of the last
print('\nSample GPT-3.5 answer:')
display(out_df['chatgpt_answers'][len(out_df) - 2])
print('\nSample GPT5-mini answer:')
display(out_df['gpt5mini_answer'][len(out_df) - 2])

Sample question:


'Are BP of 119/65 and pulse of 35 causes for concern?Yesterday I was very dizzy, short of breath, heart rate around 35,BP 119/65. I am 51,female, overweight and have RA and Fibermyalgia. Do you know what might have caused this episode yesterday? I figure I would not feel okay today if it had been a heart attack.'


Sample GPT-3.5 answer:


"['It is not uncommon for people with rheumatoid arthritis (RA) and fibromyalgia to experience symptoms such as dizziness, shortness of breath, and a low heart rate. These symptoms can be caused by a variety of factors, including inflammation, fatigue, and stress. It is possible that your symptoms yesterday were related to one or more of these factors.\\n\\nIt is important to note that a blood pressure reading of 119/65 and a pulse rate of 35 are within normal ranges and do not necessarily indicate a cause for concern. However, if you are experiencing persistent or severe symptoms, it is always a good idea to consult a healthcare professional for further evaluation and guidance. They can help determine the cause of your symptoms and provide appropriate treatment.']"


Sample GPT5-mini answer:


'A resting heart rate around 35 beats per minute is lower than normal for an adult and — especially when it’s accompanied by dizziness and shortness of breath — is a sign that needs urgent medical attention.\n\nWhy this is concerning\n- Normal resting heart rate for most adults is about 60–100 bpm. A sustained rate of 35 bpm (bradycardia) can reduce cardiac output and cause symptoms such as lightheadedness, syncope (fainting), shortness of breath, and fatigue.  \n- Your blood pressure of 119/65 is within a normal range, but a normal BP does not rule out a dangerous problem if the heart rate is very low and you are symptomatic.\n\nPossible causes (not a diagnosis)\nSeveral things can cause symptomatic bradycardia and the episode you describe:\n- Medications (common causes): beta‑blockers, nondihydropyridine calcium‑channel blockers (verapamil, diltiazem), digoxin, certain antiarrhythmics, some antidepressants or opioids. Review any prescription, over‑the‑counter, or herbal meds you take

### Check NaN counts

In [12]:
pd.isna(out_df['gpt5mini_answer']).value_counts()

,count
gpt5mini_answer,
False,21268
True,2071
